In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path

wt_raw_data_file = "/gpfs/Labs/Uzun/DATA/PROJECTS/2026.RSV.R03.CHINTAN/SC_RNA_SEQ/CELLRANGER_OUTPUTS/BATCH_2026_02/WT_Sample01/outs/filtered_feature_bc_matrix.h5"

figdir = Path("/gpfs/Home/esm5360/bulk_rna_seq_analysis/figures/Step_020_preprocess_wildtype")
figdir.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(figdir)

## Filter Wildtype Sample

### Read in the data file and remove duplicate variables

In [ ]:
wt_raw_adata = sc.read_10x_h5(wt_raw_data_file)
wt_raw_adata.var_names_make_unique()
wt_raw_adata

### Filter out low-quality cells and genes

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
wt_raw_adata.var["mt"] = wt_raw_adata.var_names.str.upper().str.startswith("MT-")
# ribosomal genes
wt_raw_adata.var["ribo"] = wt_raw_adata.var_names.str.upper().str.startswith(("RPS", "RPL"))
# hemoglobin genes
wt_raw_adata.var["hb"] = wt_raw_adata.var_names.str.upper().str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(wt_raw_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    wt_raw_adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    save="_before_processing.png",
)

In [ ]:
sc.pl.scatter(
    wt_raw_adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt", save="_scatter_pct_mt_before_processing.png"
)

In [ ]:
min_expressed_genes = 500
max_expressed_genes = 10000
min_UMI = 1000
max_UMI = 20000
max_mit_pct = 10

wt_filtered_adata = wt_raw_adata.copy()

sc.pp.filter_cells(wt_filtered_adata, min_genes=min_expressed_genes)
sc.pp.filter_cells(wt_filtered_adata, max_genes=max_expressed_genes)
sc.pp.filter_genes(wt_filtered_adata, min_cells=min_UMI)
sc.pp.filter_genes(wt_filtered_adata, max_cells=max_UMI)

wt_filtered_adata = wt_filtered_adata[wt_filtered_adata.obs["pct_counts_mt"] < max_mit_pct]

In [ ]:
sc.pl.violin(
    wt_filtered_adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    save="_after_processing.png",
)

In [ ]:
sc.pl.scatter(
    wt_filtered_adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt", save="_scatter_pct_mt_after_processing.png"
)

### Remove doublets

In [ ]:
sc.pp.scrublet(wt_filtered_adata)

In [ ]:
# Saving count data
wt_filtered_adata.layers["counts"] = wt_filtered_adata.X.copy()

### Scale / normalize / select highly-varible genes

In [ ]:
# Normalize + log (Seurat NormalizeData)
sc.pp.normalize_total(wt_filtered_adata, target_sum=1e4)
sc.pp.log1p(wt_filtered_adata)

# HVG selection (Seurat FindVariableFeatures)
sc.pp.highly_variable_genes(wt_filtered_adata, flavor='seurat', n_top_genes=2000)
sc.pl.highly_variable_genes(wt_filtered_adata, save="_hvg.png")

# Subset to HVGs
wt_filtered_adata = wt_filtered_adata[:, wt_filtered_adata.var.highly_variable]

# Scale (Seurat ScaleData)
sc.pp.scale(wt_filtered_adata, max_value=10)

### Clustering and dimensionality reduction

In [ ]:
# PCA (Seurat RunPCA)
sc.pp.pca(wt_filtered_adata, n_comps=50)

# Neighbors (Seurat FindNeighbors)
sc.pp.neighbors(wt_filtered_adata, n_neighbors=10, n_pcs=40)

# Find clusters (Seurat FindClusters)
sc.tl.leiden(wt_filtered_adata, resolution=0.1, flavor="igraph") 

# Visualize
sc.tl.umap(wt_filtered_adata)
sc.pl.umap(wt_filtered_adata, color=['leiden'])

### Color clusters by doublets / counts

In [ ]:
# Look at how the doublet scores and predicted doublets are distributed across clusters
sc.pl.umap(
    wt_filtered_adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.2,
    size=3,
    save="_doublets.png",
)

sc.pl.umap(
    wt_filtered_adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.3,
    ncols=2,
    save="_qc_metrics.png",
)

### Filter out predicted doublets

In [ ]:
wt_filtered_adata = wt_filtered_adata[wt_filtered_adata.obs["predicted_doublet"] == False]

In [ ]:
# Look at how the doublet scores and predicted doublets are distributed across clusters
sc.pl.umap(
    wt_filtered_adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.2,
    size=3,
)

sc.pl.umap(
    wt_filtered_adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.3,
    ncols=2,
)

### Check clustering resolution

In [ ]:
for res in [0.01, 0.5, 2.0]:
    sc.tl.leiden(wt_filtered_adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")
    
sc.pl.umap(
    wt_filtered_adata,
    color=["leiden_res_0.01", "leiden_res_0.50", "leiden_res_2.00"],
    legend_loc="on data",
    save="_leiden_resolution.png"
)

### Save filtered data

In [ ]:
wt_filtered_adata.write_h5ad("/gpfs/Home/esm5360/bulk_rna_seq_analysis/data/wt_filtered_adata.h5ad")